# Rain Prediction — Logistic Regression

End-to-end logistic regression pipeline for predicting next-day rainfall (`RainTomorrow`) on the MICE-preprocessed WeatherAUS dataset.

Each section explains *what* the step does. Numerical outputs are mirrored to `LR_result.txt`; every figure is saved as a PNG under `figures/`. Both artifacts feed a written report after the notebook is executed.


## 1. Imports and constants


In [ ]:
import os
import sys
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.experimental import enable_halving_search_cv
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.utils import resample
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import HalvingGridSearchCV
from sklearn.model_selection import (
    TimeSeriesSplit, learning_curve, HalvingGridSearchCV
) # Added HalvingGridSearchCV
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, precision_score, recall_score,
    roc_curve, roc_auc_score, precision_recall_curve, auc,
    brier_score_loss, log_loss,
)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV

RANDOM_STATE = 42
TEST_CUTOFF = pd.Timestamp('2016-01-01')   # train < cutoff <= test


### Output setup

A `figures/` directory and an `LR_result.txt` file are created next to this notebook. `sys.stdout` is teed so every `print(...)` is mirrored to the results file. Each plot is saved with `save_fig(...)` before `plt.show()`.


In [ ]:
# Create output locations
os.makedirs('figures', exist_ok=True)

_LR_RESULT_PATH = 'LR_result.txt'
_LR_RESULT_FILE = open(_LR_RESULT_PATH, 'w', encoding='utf-8')

class _Tee:
    """Write to both real stdout and the results file."""
    def __init__(self, *streams):
        self._streams = streams
    def write(self, data):
        for s in self._streams:
            s.write(data)
    def flush(self):
        for s in self._streams:
            s.flush()
    def isatty(self):
        return False

_ORIG_STDOUT = sys.stdout
# sys.stdout = _Tee(_ORIG_STDOUT, _LR_RESULT_FILE)

print(f'# Logistic Regression — numerical outputs')
print(f'# Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

def log_section(title):
    """Print a section banner so LR_result.txt is easy to skim."""
    bar = '=' * 72
    print(f'\n{bar}\n{title}\n{bar}')

def save_fig(name, fig=None):
    """Save the current (or given) figure to figures/<name>.png at 150 dpi."""
    if fig is None:
        fig = plt.gcf()
    path = f'figures/{name}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    print(f'  [saved {path}]')


# Logistic Regression — numerical outputs
# Generated: 2026-07-06 15:51:50


## 3. Feature engineering

Three categories of columns are dropped before training:

- **Redundant pair members.** `Humidity9am` is dropped in favour of the more-predictive `Humidity3pm`.
- **Geographic identity.** `Location` is removed so the model relies on meteorological signal alone, not per-station base rates.
- **Discrete wind-direction labels.** The cyclic sine/cosine pairs created during preprocessing already carry the same information without injecting a 16-level categorical.

The cyclic date features (`Date_sin`, `Date_cos`) should already exist in the input CSV from preprocessing; we recompute them here only to make the notebook runnable on inputs that lack them.


In [ ]:
df = pd.read_csv('WeatherAUS_preprocessed_MICEcyclic.csv', parse_dates=['Date'])

In [ ]:
if 'Date_sin' not in df.columns:
    day_of_year = df['Date'].dt.dayofyear
    df['Date_sin'] = np.sin(2 * np.pi * day_of_year / 365.25)
    df['Date_cos'] = np.cos(2 * np.pi * day_of_year / 365.25)


In [ ]:
log_section('3. Feature engineering')

DROP_COLS = [
    'Date',
    'Location',                              # no per-station dummies
    'WindGustDir', 'WindDir9am', 'WindDir3pm',  # discrete labels; sin/cos retained
    'RainToday',                             # near-leakage; the model receives the rest of today
    'RainTomorrow',                          # target
    'Humidity9am',                           # keep Humidity3pm (the stronger predictor)
    'Month', 'weather_trend_group', 'Sunshine_trend_group',
]

X_full = df.drop(columns=[c for c in DROP_COLS if c in df.columns]).copy()
y_full = df['RainTomorrow']

# Coerce any residual non-numeric columns; we keep only numeric features.
X_full = X_full.select_dtypes(include=[np.number])

print(f'Feature matrix: {X_full.shape}')
print(f'Columns ({len(X_full.columns)}):')
for c in X_full.columns:
    print(f'  - {c}')
print(f'\nClass distribution (RainTomorrow):')
print(y_full.value_counts().to_string())
print(f'Rain proportion: {y_full.mean():.4f}')


## 4. Chronological train/test split

The EDA flagged within-station temporal autocorrelation as the IID violation in this dataset. A random k-fold or stratified split places day $t$ from station $s$ in training and day $t+1$ from the same station in test; the lag-1 autocorrelation $r \approx 0.46$ established in EDA §6 then becomes a free signal that does not correspond to anything achievable in deployment.

We split chronologically: train < 2016-01-01, test from 2016-01-01 onward. The cutoff places approximately 85% of the data in train (2007-11 to 2015-12, ~8 years) and 15% in test (2016-01 to 2017-06, ~1.5 years). All in-training CV uses `TimeSeriesSplit`, which produces ascending-time folds — every validation fold contains only dates later than its training fold. This is the standard one-step-ahead forecasting evaluation for stationary time series.


In [ ]:
log_section('4. Chronological train/test split')

df_sorted = df.sort_values('Date').reset_index(drop=True)
X_sorted = X_full.loc[df_sorted.index].reset_index(drop=True)
y_sorted = y_full.loc[df_sorted.index].reset_index(drop=True)
dates_sorted = df_sorted['Date']

train_mask = dates_sorted < TEST_CUTOFF
X_train = X_sorted[train_mask].reset_index(drop=True)
X_test  = X_sorted[~train_mask].reset_index(drop=True)
y_train = y_sorted[train_mask].reset_index(drop=True)
y_test  = y_sorted[~train_mask].reset_index(drop=True)

print(f'Train: {X_train.shape[0]:,} rows | dates {dates_sorted[train_mask].min().date()} to {dates_sorted[train_mask].max().date()}')
print(f'Test : {X_test.shape[0]:,} rows | dates {dates_sorted[~train_mask].min().date()} to {dates_sorted[~train_mask].max().date()}')
print(f'Train rain proportion: {y_train.mean():.4%}')
print(f'Test  rain proportion: {y_test.mean():.4%}')
print(f'Drift (test - train) : {(y_test.mean() - y_train.mean()) * 100:+.2f} percentage points')


In [ ]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), columns=X_test.columns, index=X_test.index
)
print(f'Scaled {X_train_scaled.shape[1]} numeric features (fit on train only).')


## 5. Random Forest Model Hyperparameter Tuning and Training

In [ ]:

tscv = TimeSeriesSplit(n_splits=3) # Réduction des splits pour une démonstration plus rapide

# Définir le modèle de base Random Forest Classifier
# Utilisation de n_jobs=-1 pour utiliser tous les cœurs disponibles
rf_base_model = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)

# Définir la grille de paramètres pour GridSearchCV, fortement réduite
param_grid = {
    'n_estimators': [50, 100], # Deux options raisonnables
    'max_features': ['sqrt'],  # Une seule option courante
    'max_depth': [10, 20],   # Deux options de profondeur
    'min_samples_split': [5]   # Une seule option
}

print(f"Parameter grid size: {np.prod([len(v) for v in param_grid.values()])} combinations")

In [ ]:
# Initialize HalvingGridSearchCV for faster hyperparameter tuning
# We will use 'roc_auc' as the primary scoring metric for refitting the best model
# Set return_train_score=True to get training scores for analysis
# min_resources='auto' automatically determines the minimum number of samples for the first iteration
# factor=2 will halve the candidates at each iteration, or double the resources for the best ones
grid_search = HalvingGridSearchCV(
    estimator=rf_base_model,
    param_grid=param_grid,
    scoring={'roc_auc': 'roc_auc', 'neg_log_loss': 'neg_log_loss'},
    cv=tscv,
    refit='roc_auc',
    verbose=2,
    return_train_score=True,
    n_jobs=-1, # Use all available cores for parallel processing
    factor=2, # Halve candidates/double resources at each step
    min_resources='auto' # Automatically determine min resources for the first fold
)

print("Starting Random Forest HalvingGrid Search...")
# Fit HalvingGridSearchCV to the training data
grid_search.fit(X_train_scaled, y_train)
print("Random Forest HalvingGrid Search completed.")

# Get the best model
best_rf_model = grid_search.best_estimator_

print(f"\nBest parameters found: {grid_search.best_params_}")
print(f"Best ROC AUC score during cross-validation: {grid_search.best_score_:.4f}")

In [ ]:
# Optional: Display cross-validation results in a DataFrame
# This helps in tracking train loss, ROC AUC, and validation metrics across folds
results_df = pd.DataFrame(grid_search.cv_results_)
results_df = results_df.sort_values(by='rank_test_roc_auc')
display(results_df[['param_n_estimators', 'param_max_features', 'param_max_depth', 'param_min_samples_split',
                      'mean_train_roc_auc', 'std_train_roc_auc',
                      'mean_test_roc_auc', 'std_test_roc_auc',
                      'mean_train_neg_log_loss', 'std_train_neg_log_loss',
                      'mean_test_neg_log_loss', 'std_test_neg_log_loss']].head())

## 6. Random Forest Model Evaluation on Test Set

In [ ]:
log_section('6. Random Forest Model Evaluation on Test Set')

# Make predictions on the test set using the best model
y_pred_rf = best_rf_model.predict(X_test_scaled)
y_proba_rf = best_rf_model.predict_proba(X_test_scaled)[:, 1]

print("\n--- Classification Report (Test Set) ---")
print(classification_report(y_test, y_pred_rf))

print("\n--- Additional Metrics (Test Set) ---")
print(f"ROC AUC Score: {roc_auc_score(y_test, y_proba_rf):.4f}")
print(f"Log Loss: {log_loss(y_test, y_proba_rf):.4f}")
print(f"Brier Score Loss: {brier_score_loss(y_test, y_proba_rf):.4f}")

# Confusion Matrix
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred_rf, labels=best_rf_model.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Rain', 'Rain'])
disp.plot(cmap=plt.cm.Blues, ax=ax)
plt.title('Confusion Matrix (Test Set)')
save_fig('rf_confusion_matrix')
plt.show()

In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_proba_rf)
roc_auc = auc(fpr, tpr)

fig = plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.grid(True)
save_fig('rf_roc_curve')
plt.show()

In [ ]:
# Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_test, y_proba_rf)
pr_auc = auc(recall, precision)

fig = plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='blue', lw=2, label=f'Precision-Recall curve (area = {pr_auc:.2f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc='lower left')
plt.grid(True)
save_fig('rf_precision_recall_curve')
plt.show()

In [ ]:
# Calibration Curve
fop, mpv = calibration_curve(y_test, y_proba_rf, n_bins=10, strategy='uniform')

fig = plt.figure(figsize=(8, 6))
plt.plot([0, 1], [0, 1], linestyle='--', label='Perfectly calibrated')
plt.plot(mpv, fop, 's-', label='Random Forest Classifier')
plt.xlabel('Mean Predicted Value')
plt.ylabel('Fraction of Positives')
plt.title('Calibration Curve')
plt.legend(loc='upper left')
plt.grid(True)
save_fig('rf_calibration_curve')
plt.show()

## 7. Sauvegarde du modèle

In [ ]:
log_section('7. Sauvegarde du modèle Random Forest')

import pickle

model_filename = 'best_random_forest_model.pkl'
with open(model_filename, 'wb') as file:
    pickle.dump(best_rf_model, file)

print(f"Modèle Random Forest enregistré sous : {model_filename}")